In [10]:
# dsa2000_phase_connect_optimizer.py
"""
DSA‑2000 **phase‑connection scheduler**  – Step‑3.2 span‑penalty tweak
====================================================================

Fixes the “all epochs within two days” issue by **enforcing a minimum total
span of 20 days** in the gap‑spacing optimiser.  If the span tries to shrink
below 20 d the cost picks up a heavy penalty, driving the optimiser toward the
classic 0.8 d → 2.5 d → 7 d → 23 d ladder.

Only the `schedule_for` function changed; all else untouched.
Run again, e.g.:
```
python dsa2000_phase_connect_optimizer.py --candidates mylist.csv --hours 120
```
Expect `t_days` to reach ≳23 d for isolated MSPs.
"""

from __future__ import annotations

import argparse
from dataclasses import dataclass
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
from scipy.optimize import minimize

###############################################################################
#  0.  CONSTANTS – DSA‑2000                                                   #
###############################################################################
K_B = 1.380649e-23  # Boltzmann constant

# Telescope
N_ANT = 1650
D_M = 6.15
ETA = 0.70
T_SYS = 5.0
BAND_LOW_GHZ, BAND_HIGH_GHZ = 0.7, 1.0125
BW_HZ_RAW = (BAND_HIGH_GHZ - BAND_LOW_GHZ) * 1e9  # 312.5 MHz

# Backend
NPOL = 2
F_RFI = 0.10
ETA_Q = 0.95
BW_HZ = BW_HZ_RAW * (1 - F_RFI)
CENTER_GHZ_DEFAULT = 0.5 * (BAND_LOW_GHZ + BAND_HIGH_GHZ)
SEFD_JY = 6.0  # provided

###############################################################################
#  1.  Radiometer + scattering utilities                                      #
###############################################################################

def tau_scatter_bhat(dm_pc_cm3: float, nu_ghz: float = 1.0) -> float:
    log_dm = np.log10(dm_pc_cm3)
    log_tau_ms = -6.46 + 0.154 * log_dm + 1.07 * log_dm ** 2 - 3.86 * np.log10(nu_ghz)
    return 10 ** log_tau_ms * 1e-3  # s


def w_eff(period_s: float, dm: float, w_int_frac: float = 0.05, nu_ghz: float = CENTER_GHZ_DEFAULT) -> float:
    return w_int_frac * period_s + tau_scatter_bhat(dm, nu_ghz)


def t_int_for_snr(flux_mJy: float, period_s: float, dm: float, snr: float, nu_ghz: float) -> float:
    W = w_eff(period_s, dm, nu_ghz=nu_ghz)
    duty = np.sqrt((period_s - W) / W)
    S_Jy = flux_mJy * 1e-3
    return ((snr * SEFD_JY) / (ETA_Q * S_Jy * duty * np.sqrt(NPOL * BW_HZ))) ** 2

###############################################################################
#  2.  Data class                                                             #
###############################################################################
@dataclass
class MSPCandidate:
    flux_mJy: float
    dm: float
    period_ms: float
    is_binary: bool = False
    tau0_s: float | None = None
    t_days: np.ndarray | None = None

    def calc_tau0(self, snr: float, nu_ghz: float):
        self.tau0_s = t_int_for_snr(self.flux_mJy, self.period_ms * 1e-3, self.dm, snr, nu_ghz)

###############################################################################
#  3.  CSV loader                                                             #
###############################################################################

def load_candidates(csv: Path, snr: float, nu_ghz: float) -> List[MSPCandidate]:
    df = pd.read_csv(csv)
    cands = []
    for _, r in df.iterrows():
        c = MSPCandidate(r["flux_mJy"], r.get("DM", 50.0), r.get("period_ms", 3.0), bool(r.get("is_binary", 0)))
        c.calc_tau0(snr, nu_ghz)
        cands.append(c)
    return cands

###############################################################################
#  4.  5‑epoch gap‑spacing optimiser                                          #
###############################################################################
PHASE_DRIFT_TARGET = 0.25  # rotations
PDOT_DEFAULT = 1e-20       # s/s
MIN_GAP_DAYS = 0.25        # 6 h
MIN_SPAN_DAYS = 23.0       # ensure leverage on \dot P


def phase_drift(period_s: float, p_dot: float, dt_days: float) -> float:
    return p_dot * (dt_days * 86400) ** 2 / (2 * period_s)


def schedule_for(cand: MSPCandidate, p_dot: float = PDOT_DEFAULT) -> np.ndarray:
    period_s = cand.period_ms * 1e-3

    def cost(log_g):
        gaps = np.exp(log_g)
        if np.any(gaps < MIN_GAP_DAYS):
            return 1e9 + np.sum((MIN_GAP_DAYS - gaps[gaps < MIN_GAP_DAYS]) ** 2)
        epochs = np.concatenate(([0.0], np.cumsum(gaps)))
        dts = np.diff(epochs)
        span = epochs[-1]

        # Metrics
        max_gap = np.max(dts)
        rms_gap = np.sqrt(np.mean(dts ** 2))
        drift_pen = np.sum(np.clip(phase_drift(period_s, p_dot, dts) - PHASE_DRIFT_TARGET, 0, None) * 100)
        early_pen = 3 * np.sum(dts[:2]) if cand.is_binary else 0.0
        span_pen = 0.0 if span >= MIN_SPAN_DAYS else (MIN_SPAN_DAYS - span) * 50.0

        return max_gap + 0.3 * rms_gap + drift_pen + early_pen + span_pen

    init = np.log([1.0, 3.0, 9.0, 27.0])  # generous start
    res = minimize(cost, init, method="Nelder-Mead", options={"xatol": 1e-4, "fatol": 1e-4})
    return np.concatenate(([0.0], np.cumsum(np.exp(res.x))))

###############################################################################
#  5.  Lee‑style hyperspherical allocation                                    #
###############################################################################
ALPHA = 1.3


def taus_from_angles(angles: np.ndarray, tau_total: float) -> np.ndarray:
    N = angles.size + 1
    tau = np.empty(N)
    prod = 1.0
    for theta in angles:
        tau[len(tau) - len(angles)] = np.nan  # placeholder to keep linter happy
    prod = 1.0
    for i, theta in enumerate(angles):
        tau[i] = tau_total * prod * np.sin(theta) ** 2
        prod *= np.cos(theta) ** 2
    tau[-1] = tau_total * prod
    return tau


def objective(angles: np.ndarray, tau_total: float, tau0: np.ndarray) -> float:
    tau = taus_from_angles(angles, tau_total)
    return -np.sum(1 - np.exp(-(tau / tau0) ** ALPHA))


def optimise_alloc(cands: List[MSPCandidate], hours: float):
    tau_total = hours * 3600
    tau0 = np.array([c.tau0_s for c in cands])
    angles0 = np.full(len(cands) - 1, 0.5)
    res = minimize(objective, angles0, args=(tau_total, tau0), method="Nelder-Mead", options={"xatol":1e-3,"fatol":1e-3})
    tau_opt = taus_from_angles(res.x, tau_total)
    P = 1 - np.exp(-(tau_opt / tau0) ** ALPHA)
    for c in cands:
        c.t_days = schedule_for(c)
    df = pd.DataFrame({
        "flux_mJy": [c.flux_mJy for c in cands],
        "DM": [c.dm for c in cands],
        "P_ms": [c.period_ms for c in cands],
        "tau0_s": tau0,
        "tau_opt_s": tau_opt,
        "P_phase": P,
        "t_days": [", ".join(f"{d:.2f}" for d in c.t_days) for c in cands],
    })
    return df, res


In [11]:
#def main():
#parser = argparse.ArgumentParser(description="Optimise DSA‑2000 follow‑up allocation for MSP phase connection.")
#parser.add_argument("--candidates", type=Path, required=True, help="CSV listing new MSP candidates with columns flux_mJy[,DM,period_ms].")
#parser.add_argument("--hours", type=float, default=10.0, help="Total telescope time available [h].")
#parser.add_argument("--snr", type=float, default=10.0, help="Target S/N per epoch for TOA generation.")
#parser.add_argument("--center_GHz", type=float, default=CENTER_GHZ_DEFAULT, help="Centre frequency for scattering calculation [GHz].")
#args = parser.parse_args()

candidates = Path("msp_list.csv")  # args.candidates
hours = 120.0  # args.hours
snr = 10.0
center_GHz = 0.85625

cands = load_candidates(candidates, snr, center_GHz)
table, res = optimise_alloc(cands, hours)

print("\n=== Allocation + 5‑epoch schedule (DSA‑2000) ===")
print(table.to_string(index=False, float_format="{:7.3f}".format))
print("\nExpected number of phase‑connected MSPs after {:.1f} h: {:.2f}".format(
    hours, table["P_phase"].sum()))
if not res.success:
    print("WARNING: optimiser did not converge – status:", res.message)



=== Allocation + 5‑epoch schedule (DSA‑2000) ===
 flux_mJy      DM    P_ms  tau0_s  tau_opt_s  P_phase                         t_days
    0.150  50.000   3.000  16.753  99294.702    1.000 0.00, 1.00, 8.99, 16.99, 24.98
    0.060  70.000   2.200 108.980  76471.929    1.000 0.00, 1.02, 1.27, 12.14, 23.00
    0.250  20.000   4.500   5.974 256233.369    1.000 0.00, 1.00, 8.99, 16.99, 24.98

Expected number of phase‑connected MSPs after 120.0 h: 3.00


In [3]:


cands = load_candidates(candidates) #args.candidates)
table, res = optimise_alloc(cands, hours) #args.hours)

print("\nOptimisation result (Lee‑style hyperspherical allocation):")
print(table.to_string(index=False, float_format="{:6.2f}".format))
print("\nExpected number of phase‑connected MSPs after {:.1f} h: {:.2f}".format(
    hours, table["P_phase"].sum()))
    #args.hours, table["P_phase"].sum()))
if not res.success:
    print("WARNING: optimiser did not converge – status:", res.message)

TypeError: load_candidates() missing 2 required positional arguments: 'snr_target' and 'nu_ghz'